In [55]:
import pandas as pd 
import joblib
import json
import numpy as np
import warnings
warnings.filterwarnings("ignore")


In [56]:
test_df = pd.read_csv("../data/raw/test.csv") 

test_ids = test_df["id"]

In [57]:
test_df["stress_level"] = test_df["stress_level"].map({"Low": 0, "Medium": 1, "High": 2})


In [58]:
test_df['academic_work_impact'] = test_df['academic_work_impact'].map({"Yes": 1, "No": 0})

In [59]:
with open("../models/encoding_metadata.json") as f:
    encoding_metadata = json.load(f)

test_df["gender"] = test_df["gender"].fillna("Unknown") if "Unknown" in encoding_metadata["gender"]["categories"] else test_df["gender"]
test_df = pd.get_dummies(test_df, columns=["gender"], prefix="gender", drop_first=True)

gender_cols = [c for c in test_df.columns if c.startswith("gender_")]
test_df[gender_cols] = test_df[gender_cols].astype(int)

# Ensure test has exactly the same gender_* columns as training (add missing ones as 0)
expected_gender_cols = encoding_metadata["gender"]["resulting_columns"]
for col in expected_gender_cols:
    if col not in test_df.columns:
        test_df[col] = 0

In [60]:
from eda_helpers import fill_missing_by_correlation

for col in test_df.columns:
    missing_count = test_df[col].isnull().sum()
    if missing_count > 0:
        perc_missing = (missing_count / len(test_df)) * 100
        print(f"Column '{col}' has {missing_count} missing values ({perc_missing:.2f}%).")
        test_df, msg = fill_missing_by_correlation(test_df, col, threshold=0.0)
        print(msg)
        perc_missing_after = (test_df[col].isnull().sum() / len(test_df)) * 100
        print(f"After filling, column '{col}' has {test_df[col].isnull().sum()} missing values ({perc_missing_after:.2f}%).")
        print("\n")

Column 'age' has 17138 missing values (5.78%).
Filled 11855 missing values in 'age' using ['weekend_screen_time', 'daily_screen_time_hours'] (LinearRegression). 5283 rows still missing (predictors were also null).
After filling, column 'age' has 5283 missing values (1.78%).


Column 'daily_screen_time_hours' has 32788 missing values (11.07%).
Filled 13148 missing values in 'daily_screen_time_hours' using ['weekend_screen_time', 'social_media_hours'] (LinearRegression). 19640 rows still missing (predictors were also null).
After filling, column 'daily_screen_time_hours' has 19640 missing values (6.63%).


Column 'social_media_hours' has 47397 missing values (16.00%).
Filled 21883 missing values in 'social_media_hours' using ['daily_screen_time_hours', 'weekend_screen_time'] (LinearRegression). 25514 rows still missing (predictors were also null).
After filling, column 'social_media_hours' has 25514 missing values (8.61%).


Column 'gaming_hours' has 59420 missing values (20.05%).
Filled

In [61]:
for col in test_df.select_dtypes(include=np.number).columns:
    if test_df[col].isnull().sum() > 0:
        median_value = test_df[col].median()
        test_df[col] = test_df[col].fillna(median_value)
        print(f"Filled missing values in column '{col}' with median value {median_value}.")

Filled missing values in column 'age' with median value 26.769691072475037.
Filled missing values in column 'daily_screen_time_hours' with median value 7.77.
Filled missing values in column 'social_media_hours' with median value 2.34.
Filled missing values in column 'gaming_hours' with median value 1.36.
Filled missing values in column 'work_study_hours' with median value 2.22.
Filled missing values in column 'sleep_hours' with median value 6.801059559036819.
Filled missing values in column 'notifications_per_day' with median value 146.38367452325764.
Filled missing values in column 'app_opens_per_day' with median value 103.78719118059908.
Filled missing values in column 'weekend_screen_time' with median value 9.58.
Filled missing values in column 'stress_level' with median value 1.0.
Filled missing values in column 'academic_work_impact' with median value 0.5182697021546754.


In [62]:
remaining_missing = test_df.isnull().sum()
remaining_missing = remaining_missing[remaining_missing > 0]

if remaining_missing.empty:
    print("No missing values remain in any column.")
else:
    print("Columns still with missing values:")
    print(remaining_missing)

perc_missing_rows = (test_df.isnull().any(axis=1).sum() / len(test_df)) * 100
print(f"\nPercentage of rows with missing values: {perc_missing_rows:.2f}%")

No missing values remain in any column.

Percentage of rows with missing values: 0.00%


In [63]:
model = joblib.load("../models/xgb_addiction_model.pkl")
expected_features = model.get_booster().feature_names

X_test_final = test_df.drop(columns=["id"], errors="ignore")
X_test_final = X_test_final.reindex(columns=expected_features, fill_value=0)


In [64]:
predictions_proba = model.predict_proba(X_test_final)[:, 1]  


In [65]:
submission = pd.DataFrame({
    "id": test_ids,
    "addicted_label": predictions_proba
})
submission.head()

,id,addicted_label
0,691369,0.997058
1,691370,0.635635
2,691371,0.987922
3,691372,0.997664
4,691373,0.994104


In [66]:
submission.to_csv("../data/processed/submission.csv", index=False)